#### 13. This question should be answered using the Weekly data set, which is part of the ISLP package. This data is similar in nature to the Smarket data from this chapter’s lab, except that it contains 1,089 weekly returns for 21 years, from the beginning of 1990 to the end of 2010.
####(e) Repeat (d) using LDA.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
!pip install ISLP
from ISLP import load_data
weekly = load_data('Weekly')

In [ ]:
weekly.head()

,Year,Lag1,Lag2,Lag3,Lag4,Lag5,Volume,Today,Direction
0,1990,0.816,1.572,-3.936,-0.229,-3.484,0.154976,-0.270,Down
1,1990,-0.270,0.816,1.572,-3.936,-0.229,0.148574,-2.576,Down
2,1990,-2.576,-0.270,0.816,1.572,-3.936,0.159837,3.514,Up
3,1990,3.514,-2.576,-0.270,0.816,1.572,0.161630,0.712,Up
4,1990,0.712,3.514,-2.576,-0.270,0.816,0.153728,1.178,Up


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import confusion_matrix, accuracy_score

train_data = weekly[weekly['Year'] <= 2008]
test_data = weekly[weekly['Year'] > 2008]

X_train = train_data[['Lag2']]
y_train = train_data['Direction']

X_test = test_data[['Lag2']]
y_test = test_data['Direction']

# Fit LDA
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)

# Predict on test set
y_pred = lda.predict(X_test)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
acc = accuracy_score(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Down", "Actual Up"],
    columns=["Predicted Down", "Predicted Up"]
)

print("Confusion Matrix:\n", cm_df)
print(f"Test Accuracy: {acc:.3f}")
print(f"Test Error Rate: {1 - acc:.3f}")

Confusion Matrix:
              Predicted Down  Predicted Up
Actual Down               9            34
Actual Up                 5            56
Test Accuracy: 0.625
Test Error Rate: 0.375


####(f) Repeat (d) using QDA.

In [ ]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

qda = QuadraticDiscriminantAnalysis()
qda.fit(X_train, y_train)

# Predict on test set
qda_pred = qda.predict(X_test)

cm = confusion_matrix(y_test, qda_pred)
accuracy = accuracy_score(y_test, qda_pred)
test_error = 1 - accuracy

cm_df = pd.DataFrame(
    cm,
    index=["Actual Down", "Actual Up"],
    columns=["Predicted Down", "Predicted Up"]
)

print(cm_df)
print(f"\nAccuracy: {accuracy:.4f}")
print(f"Test error rate: {test_error:.4f}")

             Predicted Down  Predicted Up
Actual Down               0            43
Actual Up                 0            61

Accuracy: 0.5865
Test error rate: 0.4135


####(g) Repeat (d) using KNN with K=1.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
train_scaled = scaler.fit_transform(X_train)
test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(train_scaled, y_train)
y_pred = knn.predict(test_scaled)

cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
test_error = 1 - accuracy

cm_df = pd.DataFrame(
    cm,
    index=["Actual Down", "Actual Up"],
    columns=["Predicted Down", "Predicted Up"]
)

print(cm_df)
print(f"\nAccuracy: {accuracy:.4f}")
print(f"Test error rate: {test_error:.4f}")

             Predicted Down  Predicted Up
Actual Down              22            21
Actual Up                32            29

Accuracy: 0.4904
Test error rate: 0.5096


####(h) Repeat (d) using naive Bayes.

In [ ]:
from sklearn.naive_bayes import GaussianNB
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

bayes_pred = nb_model.predict(X_test)

# 計算準確率與錯誤率
acc = accuracy_score(y_test,bayes_pred)
test_error = 1-acc
cm = confusion_matrix(y_test, bayes_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual Down", "Actual Up"],
    columns=["Predicted Down", "Predicted Up"]
)

print(cm_df)
print(f"Test Accuracy: {acc:.3f}")
print(f"Test Error Rate: {test_error:.3f}")

             Predicted Down  Predicted Up
Actual Down               0            43
Actual Up                 0            61
Test Accuracy: 0.587
Test Error Rate: 0.413


####(i) Which of these methods appears to provide the best results on this data?
LDA (highest accuracy)

####(j)Experiment with different combinations of predictors, including possible transformations and interactions, for each of the methods. Report the variables, method, and associated confusion matrix that appears to provide the best results on the held out data. Note that you should also experiment with values for K in the KNN classifer.

In [ ]:
from itertools import combinations
from sklearn.linear_model import LogisticRegression
base_predictors = ['Lag1', 'Lag2', 'Lag3', 'Lag4', 'Lag5', 'Volume']

# Create interaction terms if desired
weekly['Lag1_Lag2'] = weekly['Lag1'] * weekly['Lag2']
weekly['Lag1_Lag3'] = weekly['Lag1'] * weekly['Lag3']

# --- Generate predictor combinations (2-3 variables) ---
predictor_combinations = []
for r in range(2, 4):
    predictor_combinations += list(combinations(base_predictors, r))

# Add combinations with interactions
predictor_combinations += [
    ('Lag1', 'Lag2', 'Lag1_Lag2'),
    ('Lag1', 'Lag3', 'Lag1_Lag3'),
]

# --- Split train/test (1990-2008 train, 2009-2010 test) ---
train_data = weekly[weekly['Year'] <= 2008]
test_data = weekly[weekly['Year'] > 2008]

# --- Initialize best results dictionary ---
best_results = {
    'Logistic': {'acc': 0},
    'LDA': {'acc': 0},
    'QDA': {'acc': 0},
    'KNN': {'acc': 0, 'K': None}
}

scaler = StandardScaler()

# --- Loop through predictor combinations ---
for combo in predictor_combinations:
    X_train = train_data[list(combo)]
    X_test = test_data[list(combo)]
    y_train = train_data['Direction']
    y_test = test_data['Direction']

    # --- Logistic Regression ---
    logreg = LogisticRegression(max_iter=1000)
    logreg.fit(X_train, y_train)
    y_pred = logreg.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    if acc > best_results['Logistic']['acc']:
        best_results['Logistic'].update({'acc': acc, 'predictors': combo, 'cm': confusion_matrix(y_test, y_pred)})

    # --- LDA ---
    lda = LinearDiscriminantAnalysis()
    lda.fit(X_train, y_train)
    y_pred = lda.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    if acc > best_results['LDA']['acc']:
        best_results['LDA'].update({'acc': acc, 'predictors': combo, 'cm': confusion_matrix(y_test, y_pred)})

    # --- QDA ---
    qda = QuadraticDiscriminantAnalysis()
    qda.fit(X_train, y_train)
    y_pred = qda.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    if acc > best_results['QDA']['acc']:
        best_results['QDA'].update({'acc': acc, 'predictors': combo, 'cm': confusion_matrix(y_test, y_pred)})

    # --- KNN ---
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    for K in range(1, 21):
        knn = KNeighborsClassifier(n_neighbors=K)
        knn.fit(X_train_scaled, y_train)
        y_pred = knn.predict(X_test_scaled)
        acc = accuracy_score(y_test, y_pred)
        if acc > best_results['KNN']['acc']:
            best_results['KNN'].update({'acc': acc, 'predictors': combo, 'cm': confusion_matrix(y_test, y_pred), 'K': K})

# --- Display Best Results ---
for method, res in best_results.items():
    print(f"\n--- {method} ---")
    print("Best Predictors:", res['predictors'])
    if method == 'KNN':
        print("Best K:", res['K'])
    print("Test Accuracy:", round(res['acc'], 3))
    print("Confusion Matrix:\n", pd.DataFrame(res['cm'], index=["Actual Down","Actual Up"], columns=["Predicted Down","Predicted Up"]))



--- Logistic ---
Best Predictors: ('Lag2', 'Lag3')
Test Accuracy: 0.625
Confusion Matrix:
              Predicted Down  Predicted Up
Actual Down               8            35
Actual Up                 4            57

--- LDA ---
Best Predictors: ('Lag2', 'Lag3')
Test Accuracy: 0.625
Confusion Matrix:
              Predicted Down  Predicted Up
Actual Down               8            35
Actual Up                 4            57

--- QDA ---
Best Predictors: ('Lag1', 'Lag3')
Test Accuracy: 0.615
Confusion Matrix:
              Predicted Down  Predicted Up
Actual Down              10            33
Actual Up                 7            54

--- KNN ---
Best Predictors: ('Lag1', 'Lag2', 'Lag3')
Best K: 19
Test Accuracy: 0.625
Confusion Matrix:
              Predicted Down  Predicted Up
Actual Down              23            20
Actual Up                19            42
